<a href="https://colab.research.google.com/github/usama488/Data-science/blob/main/Deep_Learning%E2%80%93Based_Disease_Risk_Prediction_Using_Clinical_Healthcare_Data_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 align="center"><b> Disease Risk Prediction using Clinical Healthcare Data: A Complete EDA, ML, DL & NLP Pipeline</b></h1>

# The goal of this project is to predict disease risk using patient clinical data through a comprehensive data science pipeline. The study integrates data preprocessing, Exploratory Data Analysis (EDA), feature engineering, Machine Learning models, Deep Learning techniques with epoch tracking, Natural Language Processing (NLP) for feature analysis, and model interpretability using SHAP. The final system is designed to be deployment-ready through an interactive Streamlit application for real-time predictions.
</p>

#  1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'colab'
from scipy.stats import ttest_ind, chi2_contingency
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, classification_report)
import xgboost as xgb
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, Callback
from sklearn.feature_extraction.text import TfidfVectorizer
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import shap
import time
import warnings
warnings.filterwarnings('ignore')
import joblib
print(" All libraries imported.")

#  2. Load the Dataset

In [ ]:
df = pd.read_csv("/content/NACC_APOE_CVD_filtered.csv")

In [ ]:
df.head()

,NACCID,SEX,BIRTHYR,NACCAPOE,DEMENTED,CVHATT,HATTMULT,CVAFIB,CVANGIO,CVBYPASS,...,STROKE,STROKIF,STROKDEC,STKIMAG,CVD,CVDIF,VASC,VASCIF,VASCPS,VASCPSIF
0,NACC000011,2,1944,1.0,0,0.0,NaN,0.0,0.0,0.0,...,0.0,7.0,NaN,NaN,NaN,NaN,0.0,7.0,NaN,NaN
1,NACC000034,2,1935,4.0,0,0.0,8.0,0.0,0.0,0.0,...,NaN,NaN,8.0,8.0,0.0,7.0,NaN,NaN,NaN,NaN
2,NACC000067,1,1952,1.0,0,0.0,NaN,0.0,0.0,0.0,...,0.0,7.0,NaN,NaN,NaN,NaN,0.0,7.0,0.0,7.0
3,NACC000095,1,1926,2.0,1,0.0,NaN,0.0,0.0,0.0,...,0.0,7.0,NaN,NaN,NaN,NaN,0.0,7.0,0.0,7.0
4,NACC000144,1,1930,1.0,0,0.0,NaN,1.0,0.0,0.0,...,0.0,8.0,NaN,NaN,NaN,NaN,8.0,8.0,8.0,8.0


In [ ]:
df.shape

(40686, 43)

In [ ]:
df.describe()

,SEX,BIRTHYR,NACCAPOE,DEMENTED,CVHATT,HATTMULT,CVAFIB,CVANGIO,CVBYPASS,CVPACDEF,...,STROKE,STROKIF,STROKDEC,STKIMAG,CVD,CVDIF,VASC,VASCIF,VASCPS,VASCPSIF
count,40686.000000,40686.000000,40686.000000,40686.000000,40686.000000,40686.000000,40686.000000,40686.000000,40686.000000,40686.000000,...,40686.000000,40686.000000,40686.000000,40686.000000,40686.000000,40686.000000,40686.000000,40686.000000,40686.000000,40686.000000
mean,1.565969,1940.892912,1.819471,0.359878,0.080519,7.949221,0.073047,0.086516,0.057612,0.003883,...,0.022416,7.103131,7.932016,7.951826,0.035467,7.059038,1.509414,7.136509,1.110505,7.077889
std,0.495635,12.652103,1.061846,0.479970,0.383839,0.627992,0.314919,0.384199,0.325891,0.069653,...,0.148033,0.763639,0.717038,0.585801,0.184959,1.019874,3.119810,0.675195,2.752139,0.676474
min,1.000000,1896.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000
25%,1.000000,1932.000000,1.000000,0.000000,0.000000,8.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,7.000000,8.000000,8.000000,0.000000,7.000000,0.000000,7.000000,0.000000,7.000000
50%,2.000000,1941.000000,2.000000,0.000000,0.000000,8.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,7.000000,8.000000,8.000000,0.000000,7.000000,0.000000,7.000000,0.000000,7.000000
75%,2.000000,1949.000000,2.000000,1.000000,0.000000,8.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,7.000000,8.000000,8.000000,0.000000,7.000000,0.000000,7.000000,0.000000,7.000000
max,2.000000,2003.000000,6.000000,1.000000,2.000000,8.000000,2.000000,2.000000,2.000000,2.000000,...,1.000000,8.000000,8.000000,8.000000,1.000000,8.000000,8.000000,8.000000,8.000000,8.000000


In [ ]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
SEX,40686.0,1.565969,0.495635,1.0,1.0,2.0,2.0,2.0
BIRTHYR,40686.0,1940.892912,12.652103,1896.0,1932.0,1941.0,1949.0,2003.0
NACCAPOE,40686.0,1.819471,1.061846,1.0,1.0,2.0,2.0,6.0
DEMENTED,40686.0,0.359878,0.479970,0.0,0.0,0.0,1.0,1.0
CVHATT,40686.0,0.080519,0.383839,0.0,0.0,0.0,0.0,2.0
HATTMULT,40686.0,7.949221,0.627992,0.0,8.0,8.0,8.0,8.0
CVAFIB,40686.0,0.073047,0.314919,0.0,0.0,0.0,0.0,2.0
CVANGIO,40686.0,0.086516,0.384199,0.0,0.0,0.0,0.0,2.0
CVBYPASS,40686.0,0.057612,0.325891,0.0,0.0,0.0,0.0,2.0
CVPACDEF,40686.0,0.003883,0.069653,0.0,0.0,0.0,0.0,2.0


In [ ]:
df.columns

Index(['NACCID', 'SEX', 'BIRTHYR', 'NACCAPOE', 'DEMENTED', 'CVHATT',
       'HATTMULT', 'CVAFIB', 'CVANGIO', 'CVBYPASS', 'CVPACDEF', 'CVPACE',
       'CVCHF', 'CVANGINA', 'CVHVALVE', 'CVOTHR', 'CVOTHRX', 'MYOINF',
       'CONGHRT', 'AFIBRILL', 'ANGINA', 'ANGIOCP', 'ANGIOPCI', 'PACEMAKE',
       'HVALVE', 'CBSTROKE', 'STROKMUL', 'NACCSTYR', 'CBTIA', 'TIAMULT',
       'NACCTIYR', 'HXSTROKE', 'PREVSTK', 'STROKE', 'STROKIF', 'STROKDEC',
       'STKIMAG', 'CVD', 'CVDIF', 'VASC', 'VASCIF', 'VASCPS', 'VASCPSIF'],
      dtype='object')

# 3. Data Cleaning & Overview

In [ ]:
df.info()

In [ ]:
df.drop_duplicates(inplace=True)
df.fillna(df.median(numeric_only=True), inplace=True)

In [ ]:
df.isnull().sum()

,0
NACCID,0
SEX,0
BIRTHYR,0
NACCAPOE,0
DEMENTED,0
CVHATT,0
HATTMULT,0
CVAFIB,0
CVANGIO,0
CVBYPASS,0


# 4. Exploratory Data Analysis (EDA)

# 4.1 Target Distribution

In [ ]:
fig = px.pie(df, names='Outcome', title='Diabetes Distribution (1=Diabetic, 0=Non-Diabetic)',
             color_discrete_sequence=['lightgreen','coral'])
fig.show()

# 4.2 Age Distribution by Outcome

In [ ]:
fig = px.histogram(df, x='Age', color='Outcome', barmode='overlay', opacity=0.6,
                   title='Age Distribution by Diabetes Status')
fig.show()

# 4.3 Glucose Distribution by Outcome

In [ ]:
fig = px.box(df, x='Outcome', y='Glucose', color='Outcome',
             title='Glucose Level by Diabetes Status')
fig.show()

# 4.4 BMI Distribution by Outcome

In [ ]:
fig = px.violin(df, x='Outcome', y='BMI', box=True,
                title='BMI Distribution by Diabetes Status')
fig.show()

# 4.5 Blood Pressure Analysis

In [ ]:
fig = px.box(df, x='Outcome', y='BloodPressure', color='Outcome',
             title='Blood Pressure by Diabetes Status')
fig.show()

# 4.6 Insulin Levels by Outcome

In [ ]:
fig = px.histogram(df, x='Insulin', color='Outcome', barmode='overlay', opacity=0.6,
                   title='Insulin Distribution by Diabetes Status')
fig.show()

# 4.7 Pregnancies Count by Outcome

In [ ]:
fig = px.histogram(df, x='Pregnancies', color='Outcome', barmode='group',
                   title='Number of Pregnancies by Diabetes Status')
fig.show()

# 4.8 Diabetes Pedigree Function Distribution

In [ ]:
fig = px.box(df, x='Outcome', y='DiabetesPedigreeFunction', color='Outcome',
             title='Diabetes Pedigree Function by Outcome')
fig.show()

# 4.9 Correlation Heatmap

In [ ]:
corr = df.corr()
fig = px.imshow(corr, text_auto=True, title='Correlation Heatmap',
                color_continuous_scale='RdBu', zmin=-1, zmax=1)
fig.show()

# 4.10 Scatter Matrix of Key Features

In [ ]:
key_features = ['Glucose', 'BMI', 'Age', 'Insulin', 'Outcome']
fig = px.scatter_matrix(df[key_features], dimensions=key_features[:-1], color='Outcome',
                        title='Scatter Matrix of Key Clinical Features')
fig.update_traces(diagonal_visible=False)
fig.show()

# 5. Statistical Testing

# 5.1 T‑test for Continuous Features

In [ ]:
continuous_features = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'Age', 'DiabetesPedigreeFunction']
ttest_results = []
for col in continuous_features:
    group1 = df[df['Outcome']==1][col]
    group0 = df[df['Outcome']==0][col]
    t_stat, p_val = ttest_ind(group1, group0, equal_var=False)
    ttest_results.append({'Feature': col, 'T-statistic': t_stat, 'P-value': p_val})
ttest_df = pd.DataFrame(ttest_results).sort_values('P-value')
ttest_df['Significant'] = ttest_df['P-value'] < 0.05
print(ttest_df)

# 5.2 Chi‑square Test for Pregnancies (categorical)

In [ ]:
# Create pregnancy groups
df['Preg_Group'] = pd.cut(df['Pregnancies'], bins=[-1,0,2,5,10,20], labels=['0','1-2','3-5','6-10','11+'])
ct = pd.crosstab(df['Preg_Group'], df['Outcome'])
chi2, p, dof, exp = chi2_contingency(ct)
print(f"Pregnancy groups vs Outcome: Chi-square p-value = {p:.4f}")

# 6. Feature Engineering & Preprocessing

In [ ]:
# Drop temporary column
df.drop('Preg_Group', axis=1, inplace=True)

# Create interaction features
df['Glucose_BMI'] = df['Glucose'] * df['BMI']
df['Age_Glucose'] = df['Age'] * df['Glucose']
df['BMI_Age'] = df['BMI'] * df['Age']

# Create age groups
bins = [20, 30, 40, 50, 60, 100]
labels = ['20-30', '30-40', '40-50', '50-60', '60+']
df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels)
age_dummies = pd.get_dummies(df['AgeGroup'], prefix='Age')
df = pd.concat([df, age_dummies], axis=1)

In [ ]:
# Define features and target
feature_cols = [col for col in df.columns if col not in ['Outcome', 'AgeGroup']]
X = df[feature_cols]
y = df['Outcome']
print(f"Features shape: {X.shape}")

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 7. Machine Learning Models

# 7.1 Define All Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'XGBoost': xgb.XGBClassifier(eval_metric='logloss', random_state=42)Disease Risk Prediction using Clinical Healthcare Data: A Complete EDA, ML, DL & NLP Pipeline
The goal of this project is to predict disease risk using patient clinical data through a comprehensive data science pipeline. The study integrates data preprocessing, Exploratory Data Analysis (EDA), feature engineering, Machine Learning models, Deep Learning techniques with epoch tracking, Natural Language Processing (NLP) for feature analysis, and model interpretability using SHAP. The final system is designed to be deployment-ready through an interactive Streamlit application for real-time predictions.
1. Import Libraries

[ ]
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'colab'
from scipy.stats import ttest_ind, chi2_contingency
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, classification_report)
import xgboost as xgb
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, Callback
from sklearn.feature_extraction.text import TfidfVectorizer
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import shap
import time
import warnings
warnings.filterwarnings('ignore')
import joblib
print(" All libraries imported.")
2. Load the Dataset

[ ]
df = pd.read_csv("/content/NACC_APOE_CVD_filtered.csv")

[ ]
df.head()


[ ]
df.shape
(40686, 43)

[ ]
df.describe()


[ ]
df.describe().T


[ ]
df.columns
Index(['NACCID', 'SEX', 'BIRTHYR', 'NACCAPOE', 'DEMENTED', 'CVHATT',
       'HATTMULT', 'CVAFIB', 'CVANGIO', 'CVBYPASS', 'CVPACDEF', 'CVPACE',
       'CVCHF', 'CVANGINA', 'CVHVALVE', 'CVOTHR', 'CVOTHRX', 'MYOINF',
       'CONGHRT', 'AFIBRILL', 'ANGINA', 'ANGIOCP', 'ANGIOPCI', 'PACEMAKE',
       'HVALVE', 'CBSTROKE', 'STROKMUL', 'NACCSTYR', 'CBTIA', 'TIAMULT',
       'NACCTIYR', 'HXSTROKE', 'PREVSTK', 'STROKE', 'STROKIF', 'STROKDEC',
       'STKIMAG', 'CVD', 'CVDIF', 'VASC', 'VASCIF', 'VASCPS', 'VASCPSIF'],
      dtype='object')
3. Data Cleaning & Overview

[ ]
df.info()

[ ]
df.drop_duplicates(inplace=True)
df.fillna(df.median(numeric_only=True), inplace=True)

[ ]
df.isnull().sum()

4. Exploratory Data Analysis (EDA)
4.1 Target Distribution

[ ]
fig = px.pie(df, names='Outcome', title='Diabetes Distribution (1=Diabetic, 0=Non-Diabetic)',
             color_discrete_sequence=['lightgreen','coral'])
fig.show()
4.2 Age Distribution by Outcome

[ ]
fig = px.histogram(df, x='Age', color='Outcome', barmode='overlay', opacity=0.6,
                   title='Age Distribution by Diabetes Status')
fig.show()
4.3 Glucose Distribution by Outcome

[ ]
fig = px.box(df, x='Outcome', y='Glucose', color='Outcome',
             title='Glucose Level by Diabetes Status')
fig.show()
4.4 BMI Distribution by Outcome

[ ]
fig = px.violin(df, x='Outcome', y='BMI', box=True,
                title='BMI Distribution by Diabetes Status')
fig.show()
4.5 Blood Pressure Analysis

[ ]
fig = px.box(df, x='Outcome', y='BloodPressure', color='Outcome',
             title='Blood Pressure by Diabetes Status')
fig.show()
4.6 Insulin Levels by Outcome

[ ]
fig = px.histogram(df, x='Insulin', color='Outcome', barmode='overlay', opacity=0.6,
                   title='Insulin Distribution by Diabetes Status')
fig.show()
4.7 Pregnancies Count by Outcome

[ ]
fig = px.histogram(df, x='Pregnancies', color='Outcome', barmode='group',
                   title='Number of Pregnancies by Diabetes Status')
fig.show()
4.8 Diabetes Pedigree Function Distribution

[ ]
fig = px.box(df, x='Outcome', y='DiabetesPedigreeFunction', color='Outcome',
             title='Diabetes Pedigree Function by Outcome')
fig.show()
4.9 Correlation Heatmap

[ ]
corr = df.corr()
fig = px.imshow(corr, text_auto=True, title='Correlation Heatmap',
                color_continuous_scale='RdBu', zmin=-1, zmax=1)
fig.show()
4.10 Scatter Matrix of Key Features

[ ]
key_features = ['Glucose', 'BMI', 'Age', 'Insulin', 'Outcome']
fig = px.scatter_matrix(df[key_features], dimensions=key_features[:-1], color='Outcome',
                        title='Scatter Matrix of Key Clinical Features')
fig.update_traces(diagonal_visible=False)
fig.show()
5. Statistical Testing
5.1 T‑test for Continuous Features

[ ]
continuous_features = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'Age', 'DiabetesPedigreeFunction']
ttest_results = []
for col in continuous_features:
    group1 = df[df['Outcome']==1][col]
    group0 = df[df['Outcome']==0][col]
    t_stat, p_val = ttest_ind(group1, group0, equal_var=False)
    ttest_results.append({'Feature': col, 'T-statistic': t_stat, 'P-value': p_val})
ttest_df = pd.DataFrame(ttest_results).sort_values('P-value')
ttest_df['Significant'] = ttest_df['P-value'] < 0.05
print(ttest_df)
5.2 Chi‑square Test for Pregnancies (categorical)

[ ]
# Create pregnancy groups
df['Preg_Group'] = pd.cut(df['Pregnancies'], bins=[-1,0,2,5,10,20], labels=['0','1-2','3-5','6-10','11+'])
ct = pd.crosstab(df['Preg_Group'], df['Outcome'])
chi2, p, dof, exp = chi2_contingency(ct)
print(f"Pregnancy groups vs Outcome: Chi-square p-value = {p:.4f}")
6. Feature Engineering & Preprocessing

[ ]
# Drop temporary column
df.drop('Preg_Group', axis=1, inplace=True)

# Create interaction features
df['Glucose_BMI'] = df['Glucose'] * df['BMI']
df['Age_Glucose'] = df['Age'] * df['Glucose']
df['BMI_Age'] = df['BMI'] * df['Age']

# Create age groups
bins = [20, 30, 40, 50, 60, 100]
labels = ['20-30', '30-40', '40-50', '50-60', '60+']
df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels)
age_dummies = pd.get_dummies(df['AgeGroup'], prefix='Age')
df = pd.concat([df, age_dummies], axis=1)

[ ]
# Define features and target
feature_cols = [col for col in df.columns if col not in ['Outcome', 'AgeGroup']]
X = df[feature_cols]
y = df['Outcome']
print(f"Features shape: {X.shape}")

[ ]
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

[ ]
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
7. Machine Learning Models
7.1 Define All Models

[ ]
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'XGBoost': xgb.XGBClassifier(eval_metric='logloss', random_state=42)
}
7.2 Train & Evaluate

[ ]
results = {}
predictions = {}
probabilities = {}
trained_models = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:,1]
    predictions[name] = y_pred

7.3 Model Comparison Bar Chart

[ ]
res_df = pd.DataFrame(results).T.reset_index().rename(columns={'index': 'Model'})
fig = px.bar(res_df, x='Model', y=['Accuracy', 'AUC'], barmode='group',
             title='Model Performance Comparison')
fig.show()
7.4 Confusion Matrix (Best Model)

[ ]
best_model_name = res_df.loc[res_df['AUC'].idxmax(), 'Model']
best_model = trained_models[best_model_name]
y_pred_best = predictions[best_model_name]

cm = confusion_matrix(y_test, y_pred_best)
fig = px.imshow(cm, text_auto=True, title=f'Confusion Matrix – {best_model_name}',
                x=['No Diabetes', 'Diabetes'], y=['No Diabetes', 'Diabetes'])
fig.show()
7.5 ROC Curves for All Models

[ ]
fig = go.Figure()
for name, prob in probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc_val = roc_auc_score(y_test, prob)
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name=f'{name} (AUC={auc_val:.3f})'))
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Random', line=dict(dash='dash')))
fig.update_layout(title='ROC Curves – All Models')
fig.show()
7.6 Cross‑Validation

[ ]
cv_scores = cross_val_score(best_model, X_train_scaled, y_train, cv=5, scoring='accuracy')
print(f"5-fold CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
7.7 Hyperparameter Tuning (Random Forest)

[ ]
param_grid = {'n_estimators': [50, 100, 150], 'max_depth': [5, 10, None]}
grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring='roc_auc')
grid.fit(X_train_scaled, y_train)
print(f"Best parameters: {grid.best_params_}")
print(f"Best CV AUC: {grid.best_score_:.4f}")
8. Deep Learning with Epoch Tracking
8.1 Custom Epoch Callback

[ ]
class EpochLogger(Callback):
    def __init__(self):
        self.epoch_logs = []
        self.start_time = time.time()

    def on_epoch_end(self, epoch, logs=None):
        self.epoch_logs.append({
            'epoch': epoch + 1,
            'loss': logs.get('loss'),
            'accuracy': logs.get('accuracy'),
            'val_loss': logs.get('val_loss'),
            'val_accuracy': logs.get('val_accuracy'),
            'time': time.time() - self.start_time
        })
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}: loss={logs['loss']:.4f}, acc={logs['accuracy']:.4f}, "
                  f"val_loss={logs['val_loss']:.4f}, val_acc={logs['val_accuracy']:.4f}")

epoch_logger = EpochLogger()
8.2 Build Neural Network

[ ]
nn_model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

nn_model.compile(optimizer=Adam(learning_rate=0.001),
                 loss='binary_crossentropy',
                 metrics=['accuracy'])

print(nn_model.summary())
8.3 Train with Epoch Tracking

[ ]
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
lr_reducer = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)

history = nn_model.fit(X_train_scaled, y_train,
                       epochs=100,
                       batch_size=32,
                       validation_split=0.2,
                       callbacks=[early_stop, lr_reducer, epoch_logger],
                       verbose=0)

print(f"\n Neural Network trained. Completed epochs: {len(history.history['loss'])}")
8.4 Plot Training History

[ ]
fig = make_subplots(rows=1, cols=2, subplot_titles=('Loss', 'Accuracy'))

fig.add_trace(go.Scatter(y=history.history['loss'], name='Train Loss'), row=1, col=1)
fig.add_trace(go.Scatter(y=history.history['val_loss'], name='Validation Loss'), row=1, col=1)
fig.add_trace(go.Scatter(y=history.history['accuracy'], name='Train Accuracy'), row=1, col=2)
fig.add_trace(go.Scatter(y=history.history['val_accuracy'], name='Validation Accuracy'), row=1, col=2)

fig.update_layout(title='Neural Network Training History (Epoch-by-Epoch)')
fig.show()
8.5 Evaluate Neural Network

[ ]
nn_proba = nn_model.predict(X_test_scaled).flatten()
nn_pred = (nn_proba >= 0.5).astype(int)
nn_acc = accuracy_score(y_test, nn_pred)
nn_auc = roc_auc_score(y_test, nn_proba)

print(f"Neural Network Test Accuracy: {nn_acc:.4f}")
print(f"Neural Network AUC: {nn_auc:.4f}")
8.6 Final Epoch Summary

[ ]
epoch_logs_df = pd.DataFrame(epoch_logger.epoch_logs)
print("\n Final Epoch Details:")
print(epoch_logs_df.tail(3))
print(f"Total Training Time: {epoch_logs_df['time'].iloc[-1]:.2f} seconds")
9. NLP on Feature Names
9.1 TF‑IDF Analysis

[ ]
feature_text = ' '.join(feature_cols)
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform([feature_text])
tfidf_scores = tfidf_matrix.toarray()[0]
terms = vectorizer.get_feature_names_out()

tfidf_df = pd.DataFrame({'Term': terms, 'Score': tfidf_scores}).sort_values('Score', ascending=False).head(15)
fig = px.bar(tfidf_df, x='Score', y='Term', orientation='h', title='TF‑IDF of Clinical Feature Terms')
fig.show()
9.2 Word Cloud

[ ]
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(feature_text)
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.title('Feature Names Word Cloud')
plt.axis('off')
plt.show()
10. Model Interpretability with SHAP

[ ]
# Use Random Forest for SHAP analysis
rf_model = trained_models['Random Forest']
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test_scaled)[1]  # Class 1 (disease)

# Summary plot
shap.summary_plot(shap_values, X_test_scaled, feature_names=feature_cols, show=False)
plt.title('SHAP Summary Plot – Random Forest')
plt.tight_layout()
plt.show()

11. Save Best Model

[ ]
joblib.dump(best_model, 'diabetes_best_model.pkl')
joblib.dump(scaler, 'diabetes_scaler.pkl')
joblib.dump(feature_cols, 'diabetes_features.pkl')
nn_model.save('diabetes_nn_model.h5')
print(" Models and preprocessing objects saved.")
12. Streamlit App Deployment

[ ]

}

# 7.2 Train & Evaluate

In [ ]:
results = {}
predictions = {}
probabilities = {}
trained_models = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:,1]
    predictions[name] = y_pred
    probabilities[name] = y_proba
    trained_models[name] = model

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    results[name] = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1, 'AUC': auc}
    print(f"{name:20} | Acc: {acc:.4f} | AUC: {auc:.4f}")

# 7.3 Model Comparison Bar Chart

In [ ]:
res_df = pd.DataFrame(results).T.reset_index().rename(columns={'index': 'Model'})
fig = px.bar(res_df, x='Model', y=['Accuracy', 'AUC'], barmode='group',
             title='Model Performance Comparison')
fig.show()

# 7.4 Confusion Matrix (Best Model)

In [ ]:
best_model_name = res_df.loc[res_df['AUC'].idxmax(), 'Model']
best_model = trained_models[best_model_name]
y_pred_best = predictions[best_model_name]

cm = confusion_matrix(y_test, y_pred_best)
fig = px.imshow(cm, text_auto=True, title=f'Confusion Matrix – {best_model_name}',
                x=['No Diabetes', 'Diabetes'], y=['No Diabetes', 'Diabetes'])
fig.show()

# 7.5 ROC Curves for All Models

In [ ]:
fig = go.Figure()
for name, prob in probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc_val = roc_auc_score(y_test, prob)
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name=f'{name} (AUC={auc_val:.3f})'))
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Random', line=dict(dash='dash')))
fig.update_layout(title='ROC Curves – All Models')
fig.show()

# 7.6 Cross‑Validation

In [ ]:
cv_scores = cross_val_score(best_model, X_train_scaled, y_train, cv=5, scoring='accuracy')
print(f"5-fold CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# 7.7 Hyperparameter Tuning (Random Forest)

In [ ]:
param_grid = {'n_estimators': [50, 100, 150], 'max_depth': [5, 10, None]}
grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring='roc_auc')
grid.fit(X_train_scaled, y_train)
print(f"Best parameters: {grid.best_params_}")
print(f"Best CV AUC: {grid.best_score_:.4f}")

# 8. Deep Learning with Epoch Tracking

# 8.1 Custom Epoch Callback

In [ ]:
class EpochLogger(Callback):
    def __init__(self):
        self.epoch_logs = []
        self.start_time = time.time()

    def on_epoch_end(self, epoch, logs=None):
        self.epoch_logs.append({
            'epoch': epoch + 1,
            'loss': logs.get('loss'),
            'accuracy': logs.get('accuracy'),
            'val_loss': logs.get('val_loss'),
            'val_accuracy': logs.get('val_accuracy'),
            'time': time.time() - self.start_time
        })
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}: loss={logs['loss']:.4f}, acc={logs['accuracy']:.4f}, "
                  f"val_loss={logs['val_loss']:.4f}, val_acc={logs['val_accuracy']:.4f}")

epoch_logger = EpochLogger()

# 8.2 Build Neural Network

In [ ]:
nn_model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

nn_model.compile(optimizer=Adam(learning_rate=0.001),
                 loss='binary_crossentropy',
                 metrics=['accuracy'])

print(nn_model.summary())

# 8.3 Train with Epoch Tracking

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
lr_reducer = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)

history = nn_model.fit(X_train_scaled, y_train,
                       epochs=100,
                       batch_size=32,
                       validation_split=0.2,
                       callbacks=[early_stop, lr_reducer, epoch_logger],
                       verbose=0)

print(f"\n Neural Network trained. Completed epochs: {len(history.history['loss'])}")

# 8.4 Plot Training History

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Loss', 'Accuracy'))

fig.add_trace(go.Scatter(y=history.history['loss'], name='Train Loss'), row=1, col=1)
fig.add_trace(go.Scatter(y=history.history['val_loss'], name='Validation Loss'), row=1, col=1)
fig.add_trace(go.Scatter(y=history.history['accuracy'], name='Train Accuracy'), row=1, col=2)
fig.add_trace(go.Scatter(y=history.history['val_accuracy'], name='Validation Accuracy'), row=1, col=2)

fig.update_layout(title='Neural Network Training History (Epoch-by-Epoch)')
fig.show()

# 8.5 Evaluate Neural Network

In [ ]:
nn_proba = nn_model.predict(X_test_scaled).flatten()
nn_pred = (nn_proba >= 0.5).astype(int)
nn_acc = accuracy_score(y_test, nn_pred)
nn_auc = roc_auc_score(y_test, nn_proba)

print(f"Neural Network Test Accuracy: {nn_acc:.4f}")
print(f"Neural Network AUC: {nn_auc:.4f}")

# 8.6 Final Epoch Summary

In [ ]:
epoch_logs_df = pd.DataFrame(epoch_logger.epoch_logs)
print("\n Final Epoch Details:")
print(epoch_logs_df.tail(3))
print(f"Total Training Time: {epoch_logs_df['time'].iloc[-1]:.2f} seconds")

# 9. NLP on Feature Names

# 9.1 TF‑IDF Analysis

In [ ]:
feature_text = ' '.join(feature_cols)
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform([feature_text])
tfidf_scores = tfidf_matrix.toarray()[0]
terms = vectorizer.get_feature_names_out()

tfidf_df = pd.DataFrame({'Term': terms, 'Score': tfidf_scores}).sort_values('Score', ascending=False).head(15)
fig = px.bar(tfidf_df, x='Score', y='Term', orientation='h', title='TF‑IDF of Clinical Feature Terms')
fig.show()

# 9.2 Word Cloud

In [ ]:
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(feature_text)
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.title('Feature Names Word Cloud')
plt.axis('off')
plt.show()

# 10. Model Interpretability with SHAP

In [ ]:
# Use Random Forest for SHAP analysis
rf_model = trained_models['Random Forest']
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test_scaled)[1]  # Class 1 (disease)

# Summary plot
shap.summary_plot(shap_values, X_test_scaled, feature_names=feature_cols, show=False)
plt.title('SHAP Summary Plot – Random Forest')
plt.tight_layout()
plt.show()

# Bar plot of mean absolute SHAP values
shap_mean_abs = np.abs(shap_values).mean(axis=0)
shap_df = pd.DataFrame({'Feature': feature_cols, 'SHAP_Importance': shap_mean_abs})
shap_df = shap_df.sort_values('SHAP_Importance', ascending=False).head(10)
fig = px.bar(shap_df, x='SHAP_Importance', y='Feature', orientation='h',
             title='SHAP Feature Importance – Random Forest')
fig.show()

# 11. Save Best Model

In [ ]:
joblib.dump(best_model, 'diabetes_best_model.pkl')
joblib.dump(scaler, 'diabetes_scaler.pkl')
joblib.dump(feature_cols, 'diabetes_features.pkl')
nn_model.save('diabetes_nn_model.h5')
print(" Models and preprocessing objects saved.")

# 12. Streamlit App Deployment

In [ ]:
!pip install streamlit pyngrok -q

%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib

# Load model and preprocessors
model = joblib.load('diabetes_best_model.pkl')
scaler = joblib.load('diabetes_scaler.pkl')
features = joblib.load('diabetes_features.pkl')

st.set_page_config(page_title="Diabetes Risk Predictor", layout="centered")
st.title("🩺 Diabetes Risk Prediction using Clinical Data")
st.markdown("Enter patient health metrics to predict diabetes risk.")

col1, col2 = st.columns(2)
with col1:
    pregnancies = st.number_input("Number of Pregnancies", 0, 20, 0)
    glucose = st.number_input("Glucose Level (mg/dl)", 0, 200, 100)
    blood_pressure = st.number_input("Blood Pressure (mm Hg)", 0, 150, 70)
    skin_thickness = st.number_input("Skin Thickness (mm)", 0, 100, 20)
with col2:
    insulin = st.number_input("Insulin Level (mu U/ml)", 0, 900, 80)
    bmi = st.number_input("BMI (kg/m²)", 10, 60, 25)
    dpf = st.number_input("Diabetes Pedigree Function", 0.0, 2.5, 0.5)
    age = st.number_input("Age (years)", 18, 120, 30)

# Create derived features (must match training)
glucose_bmi = glucose * bmi
age_glucose = age * glucose
bmi_age = bmi * age

# Age groups
age_group = '20-30' if age < 30 else ('30-40' if age < 40 else ('40-50' if age < 50 else ('50-60' if age < 60 else '60+')))
age_dummies = {f'Age_{g}': 0 for g in ['20-30', '30-40', '40-50', '50-60', '60+']}
age_dummies[f'Age_{age_group}'] = 1

# Build input dictionary
input_dict = {
    'Pregnancies': pregnancies, 'Glucose': glucose, 'BloodPressure': blood_pressure,
    'SkinThickness': skin_thickness, 'Insulin': insulin, 'BMI': bmi,
    'DiabetesPedigreeFunction': dpf, 'Age': age,
    'Glucose_BMI': glucose_bmi, 'Age_Glucose': age_glucose, 'BMI_Age': bmi_age,
    **age_dummies
}
input_df = pd.DataFrame([input_dict])[features]
input_scaled = scaler.transform(input_df)
prob = model.predict_proba(input_scaled)[0][1]

st.markdown("---")
if st.button("Predict Diabetes Risk"):
    st.subheader("Prediction Result")
    if prob >= 0.5:
        st.error(f" High risk of diabetes (Probability: {prob:.2%})")
        st.markdown(" **Recommendation:** Consult a healthcare provider for further evaluation.")
    else:
        st.success(f" Low risk of diabetes (Probability: {(1-prob):.2%})")
        st.markdown(" **Recommendation:** Maintain a healthy lifestyle and regular checkups.")

In [ ]:
from pyngrok import ngrok
import subprocess
import time

# Kill any existing Streamlit processes
!pkill streamlit 2>/dev/null
time.sleep(1)

# Run Streamlit in background
process = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])
time.sleep(5)

# Create public URL
public_url = ngrok.connect(8501, "http")
print(f"\n Streamlit app is running!\n Click here to open: {public_url}")
print("Keep this cell running. To stop, interrupt the kernel.")

# Project Completed

In [ ]:
print(" Disease Risk Prediction project completed successfully!")
print(" Saved files:")
print("   - diabetes_best_model.pkl (Best ML model)")
print("   - diabetes_scaler.pkl (Feature scaler)")
print("   - diabetes_features.pkl (Feature names)")
print("   - diabetes_nn_model.h5 (Neural network model)")

# dataset
We use the Pima Indians Diabetes dataset (768 patients, 8 features) because it is widely used for binary classification in healthcare and includes real clinical measurements.